In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from utils import * 

import plotly.io as pio
pio.renderers.default = "vscode" 

# For widescreen display, overrides utils.py settings
pd.set_option('display.max_columns', None)
pd.set_option("display.max_rows", None)

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

from scipy.stats import randint, uniform
import time

# Goal: Regression Model to Predict Outage Duration

**Name(s)**: Layth Marabeh, Khanh Phan, Danny Xia   
**Repository Link**: https://github.com/k-phantastic/US-Power-Outage-Analysis   
**Website Link**: https://khanhphan.com/US-Power-Outage-Analysis (Github Pages to be updated)

**Source:** https://engineering.purdue.edu/LASCI/research-data/outages/outage.xlsx  
**Data Dictionary:** https://www.sciencedirect.com/science/article/pii/S2352340918307182?via%3Dihub#t0005

### Load Cleaned Data from previous notebook

In [2]:
# Load cleaned dataset
file_path = 'data/cleaned_outage_dataset.csv'
df = pd.read_csv(file_path)
df.head()

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND,OUTAGE.START,OUTAGE.RESTORATION,MONTH.NAME
0,2011,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,2011-07-01,17:00:00,2011-07-03,20:00:00,severe weather,NaN,NaN,3060.0,NaN,70000.0,11.60,9.18,6.81,9.28,2.33e+06,2.11e+06,2.11e+06,6.56e+06,35.55,32.23,32.20,2.31e+06,276286.0,10673.0,2.60e+06,88.94,10.64,0.41,51268.0,47586.0,1.08,1.6,4802.0,274182.0,1.75,2.2,5348119,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2011-07-01 17:00:00,2011-07-03 20:00:00,Jul
1,2014,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,2014-05-11,18:38:00,2014-05-11,18:39:00,intentional attack,vandalism,NaN,1.0,NaN,NaN,12.12,9.71,6.49,9.28,1.59e+06,1.81e+06,1.89e+06,5.28e+06,30.03,34.21,35.73,2.35e+06,284978.0,9898.0,2.64e+06,88.83,10.79,0.37,53499.0,49091.0,1.09,1.9,5226.0,291955.0,1.79,2.2,5457125,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2014-05-11 18:38:00,2014-05-11 18:39:00,May
2,2010,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,2010-10-26,20:00:00,2010-10-28,22:00:00,severe weather,heavy wind,NaN,3000.0,NaN,70000.0,10.87,8.19,6.07,8.15,1.47e+06,1.80e+06,1.95e+06,5.22e+06,28.10,34.50,37.37,2.30e+06,276463.0,10150.0,2.59e+06,88.92,10.69,0.39,50447.0,47287.0,1.07,2.7,4571.0,267895.0,1.71,2.1,5310903,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2010-10-26 20:00:00,2010-10-28 22:00:00,Oct
3,2012,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,2012-06-19,04:30:00,2012-06-20,23:00:00,severe weather,thunderstorm,NaN,2550.0,NaN,68200.0,11.79,9.25,6.71,9.19,1.85e+06,1.94e+06,1.99e+06,5.79e+06,31.99,33.54,34.44,2.32e+06,278466.0,11010.0,2.61e+06,88.90,10.68,0.42,51598.0,48156.0,1.07,0.6,5364.0,277627.0,1.93,2.2,5380443,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2012-06-19 04:30:00,2012-06-20 23:00:00,Jun
4,2015,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,2015-07-18,02:00:00,2015-07-19,07:00:00,severe weather,NaN,NaN,1740.0,250.0,250000.0,13.07,10.16,7.74,10.43,2.03e+06,2.16e+06,1.78e+06,5.97e+06,33.98,36.21,29.78,2.37e+06,289044.0,9812.0,2.67e+06,88.82,10.81,0.37,54431.0,49844.0,1.09,1.7,4873.0,292023.0,1.67,2.2,5489594,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2015-07-18 02:00:00,2015-07-19 07:00:00,Jul


In [3]:
df.columns

Index(['YEAR', 'MONTH', 'U.S._STATE', 'POSTAL.CODE', 'NERC.REGION',
       'CLIMATE.REGION', 'ANOMALY.LEVEL', 'CLIMATE.CATEGORY',
       'OUTAGE.START.DATE', 'OUTAGE.START.TIME', 'OUTAGE.RESTORATION.DATE',
       'OUTAGE.RESTORATION.TIME', 'CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL',
       'HURRICANE.NAMES', 'OUTAGE.DURATION', 'DEMAND.LOSS.MW',
       'CUSTOMERS.AFFECTED', 'RES.PRICE', 'COM.PRICE', 'IND.PRICE',
       'TOTAL.PRICE', 'RES.SALES', 'COM.SALES', 'IND.SALES', 'TOTAL.SALES',
       'RES.PERCEN', 'COM.PERCEN', 'IND.PERCEN', 'RES.CUSTOMERS',
       'COM.CUSTOMERS', 'IND.CUSTOMERS', 'TOTAL.CUSTOMERS', 'RES.CUST.PCT',
       'COM.CUST.PCT', 'IND.CUST.PCT', 'PC.REALGSP.STATE', 'PC.REALGSP.USA',
       'PC.REALGSP.REL', 'PC.REALGSP.CHANGE', 'UTIL.REALGSP', 'TOTAL.REALGSP',
       'UTIL.CONTRI', 'PI.UTIL.OFUSA', 'POPULATION', 'POPPCT_URBAN',
       'POPPCT_UC', 'POPDEN_URBAN', 'POPDEN_UC', 'POPDEN_RURAL',
       'AREAPCT_URBAN', 'AREAPCT_UC', 'PCT_LAND', 'PCT_WATER_TOT',
       'PCT

In [4]:
df.shape, df.columns

((1526, 58),
 Index(['YEAR', 'MONTH', 'U.S._STATE', 'POSTAL.CODE', 'NERC.REGION',
        'CLIMATE.REGION', 'ANOMALY.LEVEL', 'CLIMATE.CATEGORY',
        'OUTAGE.START.DATE', 'OUTAGE.START.TIME', 'OUTAGE.RESTORATION.DATE',
        'OUTAGE.RESTORATION.TIME', 'CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL',
        'HURRICANE.NAMES', 'OUTAGE.DURATION', 'DEMAND.LOSS.MW',
        'CUSTOMERS.AFFECTED', 'RES.PRICE', 'COM.PRICE', 'IND.PRICE',
        'TOTAL.PRICE', 'RES.SALES', 'COM.SALES', 'IND.SALES', 'TOTAL.SALES',
        'RES.PERCEN', 'COM.PERCEN', 'IND.PERCEN', 'RES.CUSTOMERS',
        'COM.CUSTOMERS', 'IND.CUSTOMERS', 'TOTAL.CUSTOMERS', 'RES.CUST.PCT',
        'COM.CUST.PCT', 'IND.CUST.PCT', 'PC.REALGSP.STATE', 'PC.REALGSP.USA',
        'PC.REALGSP.REL', 'PC.REALGSP.CHANGE', 'UTIL.REALGSP', 'TOTAL.REALGSP',
        'UTIL.CONTRI', 'PI.UTIL.OFUSA', 'POPULATION', 'POPPCT_URBAN',
        'POPPCT_UC', 'POPDEN_URBAN', 'POPDEN_UC', 'POPDEN_RURAL',
        'AREAPCT_URBAN', 'AREAPCT_UC', 'PCT_LAND', '

### Additional EDA and Data Cleanup

* Heavy skew in our target variable, most outages last < 2 days

In [5]:
# Note duration is in minutes; most outages last < 2 days (2880 minutes)
df['OUTAGE.DURATION'].describe()

count      1468.00
mean       2582.58
std        5813.37
min           0.00
25%         100.00
50%         691.00
75%        2880.00
max      108653.00
Name: OUTAGE.DURATION, dtype: float64

In [6]:
# Plot outage duration distribution
fig = px.histogram(df, x='OUTAGE.DURATION', nbins=100, title='Distribution of Outage Durations (Minutes)')
fig.update_layout(xaxis_title='Outage Duration (Minutes)', yaxis_title='Count')
fig.show()

# Plot log outage duration distribution
fig = px.histogram(df, x=np.log1p(df['OUTAGE.DURATION']), nbins=30, title='Distribution of Log Outage Durations (Minutes)')
fig.update_layout(xaxis_title='Log Outage Duration (Minutes)', yaxis_title='Count')
fig.show()

### Further investigation on missing rows

In [7]:
# All missing rows in "cleaned" dataset: 
missing_count = df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False)
missing_count

HURRICANE.NAMES            1454
DEMAND.LOSS.MW              700
CAUSE.CATEGORY.DETAIL       467
CUSTOMERS.AFFECTED          437
OUTAGE.RESTORATION           58
OUTAGE.RESTORATION.DATE      58
OUTAGE.RESTORATION.TIME      58
OUTAGE.DURATION              58
IND.PRICE                    22
TOTAL.SALES                  22
IND.SALES                    22
COM.SALES                    22
RES.SALES                    22
TOTAL.PRICE                  22
COM.PRICE                    22
IND.PERCEN                   22
RES.PRICE                    22
COM.PERCEN                   22
RES.PERCEN                   22
POPDEN_UC                    10
POPDEN_RURAL                 10
OUTAGE.START                  9
MONTH                         9
OUTAGE.START.TIME             9
OUTAGE.START.DATE             9
CLIMATE.CATEGORY              9
ANOMALY.LEVEL                 9
MONTH.NAME                    9
CLIMATE.REGION                6
dtype: int64

In [8]:
# CAUSE.CATEGORY.DETAIL has 467 missing values, can investigate CAUSE.CATEGORY to impute missing values
print(df['CAUSE.CATEGORY'].value_counts())

# CAUSE.CATEGORY of missing CAUSE.CATEGORY.DETAIL values
print(f"\n---> Filtered by missing CAUSE.CATEGORY.DETAIL\n{df[df['CAUSE.CATEGORY.DETAIL'].isna()]['CAUSE.CATEGORY'].value_counts()}")

severe weather                   760
intentional attack               418
system operability disruption    127
public appeal                     65
equipment failure                 60
fuel supply emergency             50
islanding                         46
Name: CAUSE.CATEGORY, dtype: int64

---> Filtered by missing CAUSE.CATEGORY.DETAIL
severe weather                   187
system operability disruption     90
public appeal                     65
intentional attack                48
islanding                         46
fuel supply emergency             19
equipment failure                 12
Name: CAUSE.CATEGORY, dtype: int64


In [9]:
# Investigate missing clinmate regions --> Hawaii and Alaska only
df[df['CLIMATE.REGION'].isna()]

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND,OUTAGE.START,OUTAGE.RESTORATION,MONTH.NAME
1507,2008,12.0,Hawaii,HI,HI,NaN,-0.7,cold,2008-12-26,18:13:00,2008-12-27,17:00:00,severe weather,thunderstorm,NaN,1367.0,1060.0,294000.0,29.28,26.28,22.43,25.78,253625.0,275383.0,306397.0,835405.0,30.36,32.96,36.68,409668.0,61684.0,673.0,472025.0,86.79,13.07,0.14,50565.0,48401.0,1.04,-0.7,1646.0,67364.0,2.44,0.5,1332213,91.93,20.47,3180.8,1664.7,18.2,6.12,2.60,58.75,41.25,0.38,2008-12-26 18:13:00,2008-12-27 17:00:00,Dec
1508,2011,5.0,Hawaii,HI,PR,NaN,-0.4,normal,2011-05-02,17:06:00,2011-05-02,20:00:00,severe weather,NaN,NaN,174.0,220.0,62000.0,34.58,32.14,27.85,31.29,249369.0,292304.0,310682.0,852355.0,29.26,34.29,36.45,417531.0,60043.0,698.0,478272.0,87.30,12.55,0.15,49110.0,47586.0,1.03,0.4,1935.0,67686.0,2.86,0.6,1378227,91.93,20.47,3180.8,1664.7,18.2,6.12,2.60,58.75,41.25,0.38,2011-05-02 17:06:00,2011-05-02 20:00:00,May
1509,2006,10.0,Hawaii,HI,HECO,NaN,0.7,warm,2006-10-15,07:09:00,2006-10-15,16:12:00,severe weather,earthquake,NaN,543.0,110.0,59886.0,23.25,21.26,17.71,20.54,274609.0,307570.0,340735.0,922915.0,29.75,33.33,36.92,401592.0,61334.0,689.0,463615.0,86.62,13.23,0.15,50358.0,48909.0,1.03,1.1,1436.0,65956.0,2.18,0.5,1309731,91.93,20.47,3180.8,1664.7,18.2,6.12,2.60,58.75,41.25,0.38,2006-10-15 07:09:00,2006-10-15 16:12:00,Oct
1510,2006,6.0,Hawaii,HI,HECO,NaN,0.0,normal,2006-06-01,14:12:00,2006-06-01,18:09:00,system operability disruption,NaN,NaN,237.0,120.0,29300.0,24.09,22.06,18.74,21.45,265474.0,298487.0,327618.0,891580.0,29.78,33.48,36.75,401592.0,61334.0,689.0,463615.0,86.62,13.23,0.15,50358.0,48909.0,1.03,1.1,1436.0,65956.0,2.18,0.5,1309731,91.93,20.47,3180.8,1664.7,18.2,6.12,2.60,58.75,41.25,0.38,2006-06-01 14:12:00,2006-06-01 18:09:00,Jun
1511,2006,10.0,Hawaii,HI,HECO,NaN,0.7,warm,2006-10-15,07:09:00,2006-10-16,14:55:00,severe weather,earthquake,NaN,1906.0,1170.0,291000.0,23.25,21.26,17.71,20.54,274609.0,307570.0,340735.0,922915.0,29.75,33.33,36.92,401592.0,61334.0,689.0,463615.0,86.62,13.23,0.15,50358.0,48909.0,1.03,1.1,1436.0,65956.0,2.18,0.5,1309731,91.93,20.47,3180.8,1664.7,18.2,6.12,2.60,58.75,41.25,0.38,2006-10-15 07:09:00,2006-10-16 14:55:00,Oct
1525,2000,NaN,Alaska,AK,ASCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,equipment failure,failure,NaN,NaN,35.0,14273.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,230534.0,38074.0,854.0,273530.0,84.28,13.92,0.31,57401.0,44745.0,1.28,-2.2,724.0,36046.0,2.01,0.2,627963,66.02,21.56,1802.6,1276.0,0.4,0.05,0.02,85.76,14.24,2.90,NaN,NaN,NaN


In [10]:
# Can just use CAUSE.CATEGORY.DETAIL since it includes hurricane information
hurricanes_detail = (df['CAUSE.CATEGORY.DETAIL'] == 'hurricanes')
no_names = (df['HURRICANE.NAMES'].isna())
len(df[~no_names]), len(df[hurricanes_detail]), len(df[hurricanes_detail & no_names])

(72, 74, 2)

In [11]:
df[df['OUTAGE.DURATION'].isna()].head()

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND,OUTAGE.START,OUTAGE.RESTORATION,MONTH.NAME
22,2015,7.0,Tennessee,TN,SERC,Central,1.2,warm,2015-07-30,13:00:00,NaN,NaN,intentional attack,vandalism,NaN,NaN,0.0,0.0,10.31,10.27,7.31,9.70,4.29e+06,3.32e+06,1.89e+06,9.50e+06,45.11,34.98,19.92,2.78e+06,4.76e+05,1245.0,3.26e+06,85.35,14.61,0.04,42457.0,49844.0,0.85,1.2,1692.0,2.83e+05,0.60,0.4,6600299,66.39,12.02,1450.3,1076.2,55.6,7.05,1.72,97.84,2.16,2.16,2015-07-30 13:00:00,NaN,Jul
36,2016,7.0,Tennessee,TN,SERC,Central,-0.3,normal,2016-07-13,15:00:00,NaN,NaN,system operability disruption,public appeal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.81e+06,4.80e+05,1193.0,3.29e+06,85.39,14.58,0.04,43720.0,50660.0,0.86,3.0,1813.0,2.91e+05,0.62,0.5,6649404,66.39,12.02,1450.3,1076.2,55.6,7.05,1.72,97.84,2.16,2.16,2016-07-13 15:00:00,NaN,Jul
46,2016,4.0,Tennessee,TN,SERC,Central,1.1,warm,2016-04-27,13:36:00,NaN,NaN,intentional attack,vandalism,NaN,NaN,0.0,0.0,10.22,9.64,5.39,8.80,2.52e+06,2.58e+06,1.68e+06,6.78e+06,37.14,38.04,24.82,2.81e+06,4.80e+05,1193.0,3.29e+06,85.39,14.58,0.04,43720.0,50660.0,0.86,3.0,1813.0,2.91e+05,0.62,0.5,6649404,66.39,12.02,1450.3,1076.2,55.6,7.05,1.72,97.84,2.16,2.16,2016-04-27 13:36:00,NaN,Apr
48,2014,6.0,Wisconsin,WI,MRO,East North Central,0.0,normal,2014-06-27,13:21:00,NaN,NaN,fuel supply emergency,Coal,NaN,NaN,NaN,NaN,14.38,11.31,7.81,10.98,1.71e+06,2.01e+06,2.04e+06,5.76e+06,29.65,34.96,35.39,2.63e+06,3.46e+05,5465.0,2.98e+06,88.22,11.60,0.18,46676.0,49091.0,0.95,1.9,4680.0,2.69e+05,1.74,1.9,5759432,70.15,14.35,2123.3,1671.5,32.5,3.47,0.90,82.69,17.31,3.05,2014-06-27 13:21:00,NaN,Jun
181,2007,9.0,Texas,TX,WECC,South,-0.9,cold,2007-09-06,20:00:00,NaN,NaN,fuel supply emergency,NaN,NaN,NaN,NaN,NaN,12.44,9.88,7.76,10.30,1.33e+07,1.04e+07,9.49e+06,3.32e+07,40.12,31.26,28.60,9.17e+06,1.41e+06,168586.0,1.07e+07,85.28,13.15,1.57,49068.0,49126.0,1.00,2.5,27695.0,1.17e+06,2.37,10.9,23831983,84.70,9.35,2435.3,1539.9,15.2,3.35,0.58,97.26,2.74,2.09,2007-09-06 20:00:00,NaN,Sep


### Imputation for missing rows

In [12]:
# Imputations

def impute_climate_region(df):
    # Hawaii and Alaska don't have climate regions in dataset. 
    df.loc[df['U.S._STATE'] == 'Hawaii', 'CLIMATE.REGION'] = 'Hawaii'
    df.loc[df['U.S._STATE'] == 'Alaska', 'CLIMATE.REGION'] = 'Alaska'
    return df

def impute_cause_category_detail(df):
    # Impute based on CAUSE.CATEGORY
    fallback = {
        'severe weather':                  'Other Weather',
        'intentional attack':              'Other Intentional Attack',
        'system operability disruption':   'Other Equipment Failure',
        'equipment failure':               'Other Equipment Failure',
        'fuel supply emergency':           'Other Fuel Supply Emergency',
        'public appeal':                   'Other Public Appeal',
        'islanding':                       'Other Islanding',
    }

    df['CAUSE.CATEGORY.DETAIL'] = df['CAUSE.CATEGORY.DETAIL'].fillna(
        df['CAUSE.CATEGORY'].map(fallback)
    )
    return df

def drop_missing_months(df):
    df = df.dropna(subset=['OUTAGE.START'])
    return df

def drop_missing_price(df):
    df = df.dropna(subset=['RES.PRICE'])
    return df

def median_impute_customers_affected(df): 
    # Median imputation for CUSTOMERS.AFFECTED, grouped by CAUSE.CATEGORY.DETAIL
    df['CUSTOMERS.AFFECTED'] = df.groupby('CAUSE.CATEGORY')['CUSTOMERS.AFFECTED'].transform(
        lambda x: x.fillna(x.median())
    )
    return df

def drop_missing_durations(df):
    df = df.dropna(subset=['OUTAGE.DURATION'])
    return df

# Feature engineering
def identify_weekend(df): 
    df['OUTAGE.START'] = pd.to_datetime(df['OUTAGE.START'])
    df['START_ON_WEEKEND'] = df['OUTAGE.START'].dt.weekday.apply(lambda x: 1 if x >= 5 else 0) # 1 for Weekends, 0 for weekdays
    return df

def identify_season(df): 
    outage_month = pd.to_datetime(df['OUTAGE.START']).dt.month
    df['SEASON'] = outage_month.map({
            12: 'Winter', 1: 'Winter',  2: 'Winter',
            3:  'Spring', 4: 'Spring',  5: 'Spring',
            6:  'Summer', 7: 'Summer',  8: 'Summer',
            9:  'Fall',   10: 'Fall',   11: 'Fall',
        })
    return df

In [13]:
df_engineered = (
    df.pipe(impute_climate_region)
      .pipe(impute_cause_category_detail)
      .pipe(drop_missing_months)
      .pipe(drop_missing_price)
      .pipe(drop_missing_durations)
      .pipe(median_impute_customers_affected)
      .pipe(identify_weekend)
      .pipe(identify_season)
)

print(f"Rows dropped: {len(df) - len(df_engineered)} ({(len(df) - len(df_engineered)) / len(df) * 100:.1f}% of original)")

Rows dropped: 70 (4.6% of original)


In [14]:
# Some columns will still have missing values, but our selected features will only be using a subset
df_engineered.isna().sum()[df_engineered.isna().sum() > 0] 

HURRICANE.NAMES    1385
DEMAND.LOSS.MW      663
POPDEN_UC            10
POPDEN_RURAL         10
dtype: int64

### Setup for Model and Training

In [15]:
# Feature selection

cat_features = [
    'SEASON', 
    'U.S._STATE', 
    'NERC.REGION', 
    'CLIMATE.REGION', 
    'CLIMATE.CATEGORY',
    'CAUSE.CATEGORY',
    'CAUSE.CATEGORY.DETAIL',
]
num_features = [
    'MONTH',
    'START_ON_WEEKEND',
    'ANOMALY.LEVEL', 
    'CUSTOMERS.AFFECTED',
    'RES.PRICE',
    'COM.PRICE',
    'IND.PRICE',
    'UTIL.CONTRI',
    'PI.UTIL.OFUSA',
]

# Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

preprocessor_dense = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), cat_features)
    ]
)

features = cat_features + num_features
target = 'OUTAGE.DURATION'

In [16]:
print(f"Initial cleaned dataset shape:  {df.shape}")
print(f"Engineered dataset shape:       {df_engineered.shape}")

X = df_engineered[features]
y = np.log1p(df_engineered[target]) # Log transform target to reduce skew
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape:             {X_train.shape}, {y_train.shape}")
print(f"Test set shape:                 {X_test.shape}, {y_test.shape}")

Initial cleaned dataset shape:  (1526, 58)
Engineered dataset shape:       (1456, 60)
Training set shape:             (1164, 16), (1164,)
Test set shape:                 (292, 16), (292,)


In [17]:
# General Setup for testing different models

results_log = [] # For final dataframe creation

def plot_log_and_reg_residuals(residual_df, model_name):
    fig = make_subplots(rows=1, cols=2, subplot_titles=[
        'Residuals vs Predicted (log scale)',
        'Residuals vs Predicted (minutes)',
    ])

    fig.add_trace(
        go.Scatter(
            x=residual_df['Predicted Log Duration'],
            y=residual_df['Residuals (log)'],
            mode='markers', opacity=0.5, marker=dict(size=4),
            name='log scale',
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=residual_df['Predicted Duration (min)'],
            y=residual_df['Residuals (min)'],
            mode='markers', opacity=0.5, marker=dict(size=4),
            name='minutes',
        ),
        row=1, col=2,
    )

    # Zero lines for reference
    fig.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=1)
    fig.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=2)

    fig.update_layout(
        title=f'Residuals - {model_name}',
        title_pad=dict(t=20),
        margin=dict(t=80),
        width=1000, height=450,
        showlegend=False,
    )
    fig.update_xaxes(title_text='Predicted (log min)', row=1, col=1)
    fig.update_xaxes(title_text='Predicted (min)', row=1, col=2)
    fig.update_yaxes(title_text='Residuals', row=1, col=1)
    fig.update_yaxes(title_text='Residuals', row=1, col=2)
    fig.show()

def train_and_evaluate_model(model, X_train, y_train, X_test, y_test, label=None):
    start_time = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    y_pred_minutes = np.expm1(y_pred) # Convert back to minutes for residuals plot
    y_test_minutes = np.expm1(y_test)

    metrics = {
        'Model': label or model.__class__.__name__,
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': mean_squared_error(y_test, y_pred, squared=False),
        'MAE (minutes)': mean_absolute_error(y_test_minutes, y_pred_minutes),
        'RMSE (minutes)': mean_squared_error(y_test_minutes, y_pred_minutes, squared=False),
        'R^2': r2_score(y_test, y_pred),           
    }

    print(f"Model:                   {metrics['Model']}")
    print(f"MAE:                     {metrics['MAE']:.4f}")
    print(f"RMSE (log):              {metrics['RMSE']:.4f}")
    print(f"MAE (minutes):           {metrics['MAE (minutes)']:.2f} minutes")
    print(f"RMSE (minutes):          {metrics['RMSE (minutes)']:.2f} minutes")
    print(f"R^2:                     {metrics['R^2']:.4f}")
    print(f"Training Time:           {time.time() - start_time:.2f} seconds")
    # Plot residuals
    residual_df = pd.DataFrame({
        'Predicted Log Duration': y_pred,
        'Residuals (log)': y_test - y_pred,
        'Predicted Duration (min)': y_pred_minutes,
        'Residuals (min)': y_test_minutes - y_pred_minutes
    })
    plot_log_and_reg_residuals(residual_df, metrics['Model'])

    results_log.append(metrics)
    return model, y_pred

def train_tune_evaluate(model, param_distributions, X_train, y_train, X_test, y_test, n_iter=50, label=None):
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring='neg_mean_absolute_error',
        cv=3,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )
    print(f"Starting hyperparameter tuning for {label or model.__class__.__name__}...")
    start_time = time.time()
    random_search.fit(X_train, y_train)
    print(f"Completed in {time.time() - start_time:.2f} seconds")

    print(f"Best Hyperparameters: {random_search.best_params_}")
    best_model = random_search.best_estimator_

    tuned_label = f"{label or model.__class__.__name__} (tuned)"
    return train_and_evaluate_model(best_model, X_train, y_train, X_test, y_test, label=tuned_label)

In [18]:
# Linear Regression - Baseline
linear_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

linear_reg, linear_pred = train_and_evaluate_model(linear_pipeline, X_train, y_train, X_test, y_test, label="Linear Regression (Baseline)")

Model:                   Linear Regression (Baseline)
MAE:                     1.6012
RMSE (log):              2.0603
MAE (minutes):           2300.49 minutes
RMSE (minutes):          7015.37 minutes
R^2:                     0.4654
Training Time:           0.06 seconds


In [19]:
# Ridge Regression - Hyperparameter Tuning
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])
ridge_param_dist = {
    'regressor__alpha': np.logspace(-3, 3, 50)
}
ridge_reg, ridge_pred = train_tune_evaluate(ridge_pipeline, ridge_param_dist, X_train, y_train, X_test, y_test, label="Ridge Regression")


Starting hyperparameter tuning for Ridge Regression...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
Completed in 3.49 seconds
Best Hyperparameters: {'regressor__alpha': 8.286427728546842}
Model:                   Ridge Regression (tuned)
MAE:                     1.5692
RMSE (log):              2.0006
MAE (minutes):           1779.71 minutes
RMSE (minutes):          4279.47 minutes
R^2:                     0.4959
Training Time:           0.03 seconds


In [20]:
# Random Forest Regressor
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_dense),
    ('regressor', RandomForestRegressor(random_state=42))
])
rf_param_dist = {
    'regressor__n_estimators': randint(200, 1000),
    'regressor__max_depth': [None, 1, 2, 3, 4, 5, 10, 20, 30, 40, 50],
    'regressor__min_samples_split': randint(2, 20),
    'regressor__min_samples_leaf': randint(1, 10),
    'regressor__max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7, 1.0],
    'regressor__bootstrap': [True, False],
    'regressor__max_samples': [0.6, 0.7, 0.8, 0.9, None],
}
rf_reg, rf_pred = train_tune_evaluate(rf_pipeline, rf_param_dist, X_train, y_train, X_test, y_test, n_iter=100, label="Random Forest Regressor")

Starting hyperparameter tuning for Random Forest Regressor...
Fitting 3 folds for each of 100 candidates, totalling 300 fits


c:\Users\khpha\anaconda3\lib\site-packages\sklearn\model_selection\_validation.py:372: FitFailedWarning:


90 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
90 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\khpha\anaconda3\lib\site-packages\sklearn\model_selection\_validation.py", line 680, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\khpha\anaconda3\lib\site-packages\sklearn\pipeline.py", line 394, in fit
    self._final_estimator.fit(Xt, y, **fit_params_last_step)
  File "c:\Users\khpha\anaconda3\lib\site-packages\sklearn\ensemble\_forest.py", line 379, in fit
    raise ValueError(
ValueError: `max_sample` cannot be se

Completed in 42.00 seconds
Best Hyperparameters: {'regressor__bootstrap': True, 'regressor__max_depth': 20, 'regressor__max_features': 0.3, 'regressor__max_samples': 0.8, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 6, 'regressor__n_estimators': 689}
Model:                   Random Forest Regressor (tuned)
MAE:                     1.4079
RMSE (log):              1.8672
MAE (minutes):           1596.33 minutes
RMSE (minutes):          4108.08 minutes
R^2:                     0.5609
Training Time:           2.62 seconds


In [21]:
# XGBoost Regressor
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_dense),
    ('regressor', XGBRegressor(random_state=42, objective='reg:squarederror'))
])

xgb_param_dist = {
    'regressor__n_estimators': randint(200, 1000),
    'regressor__learning_rate': uniform(0.01, 0.2),
    'regressor__max_depth': randint(3, 10),
    'regressor__subsample': uniform(0.6, 0.4),
    'regressor__colsample_bytree': uniform(0.4, 0.6),
    'regressor__min_child_weight': randint(1, 10),
    'regressor__gamma': uniform(0, 0.5),
    'regressor__reg_alpha': uniform(0, 1),
    'regressor__reg_lambda': uniform(0.5, 2),
}

xgb_reg, xgb_pred = train_tune_evaluate(xgb_pipeline, xgb_param_dist, X_train, y_train, X_test, y_test, n_iter=100, label="XGBoost Regressor")

Starting hyperparameter tuning for XGBoost Regressor...
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Completed in 20.82 seconds
Best Hyperparameters: {'regressor__colsample_bytree': 0.4530955012311517, 'regressor__gamma': 0.0979914312095726, 'regressor__learning_rate': 0.019045457782107613, 'regressor__max_depth': 7, 'regressor__min_child_weight': 2, 'regressor__n_estimators': 252, 'regressor__reg_alpha': 0.5867511656638482, 'regressor__reg_lambda': 2.430510614528276, 'regressor__subsample': 0.8428136990746738}
Model:                   XGBoost Regressor (tuned)
MAE:                     1.4420
RMSE (log):              1.9074
MAE (minutes):           1595.25 minutes
RMSE (minutes):          4082.89 minutes
R^2:                     0.5418
Training Time:           0.34 seconds


In [22]:
# HistGradientBoostingRegressor 
from sklearn.ensemble import HistGradientBoostingRegressor
hgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_dense),
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])

hgb_param_dist = {
    'regressor__learning_rate':    uniform(0.01, 0.2),
    'regressor__max_iter':         randint(100, 500),
    'regressor__max_depth':        randint(3, 10),
    'regressor__min_samples_leaf': randint(10, 50),
    'regressor__l2_regularization': uniform(0, 1),
    'regressor__max_leaf_nodes':   randint(20, 100),
}

hgb_reg, hgb_pred = train_tune_evaluate(hgb_pipeline, hgb_param_dist, X_train, y_train, X_test, y_test, n_iter=100, label="HistGradientBoostingRegressor")

Starting hyperparameter tuning for HistGradientBoostingRegressor...
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Completed in 94.68 seconds
Best Hyperparameters: {'regressor__l2_regularization': 0.09179906581344188, 'regressor__learning_rate': 0.028831397653712024, 'regressor__max_depth': 8, 'regressor__max_iter': 337, 'regressor__max_leaf_nodes': 81, 'regressor__min_samples_leaf': 14}
Model:                   HistGradientBoostingRegressor (tuned)
MAE:                     1.4731
RMSE (log):              1.9597
MAE (minutes):           1598.56 minutes
RMSE (minutes):          4010.43 minutes
R^2:                     0.5163
Training Time:           10.04 seconds


In [23]:
results_df = pd.DataFrame(results_log)
results_df

,Model,MAE,RMSE,MAE (minutes),RMSE (minutes),R^2
0,Linear Regression (Baseline),1.60,2.06,2300.49,7015.37,0.47
1,Ridge Regression (tuned),1.57,2.00,1779.71,4279.47,0.50
2,Random Forest Regressor (tuned),1.41,1.87,1596.33,4108.08,0.56
3,XGBoost Regressor (tuned),1.44,1.91,1595.25,4082.89,0.54
4,HistGradientBoostingRegressor (tuned),1.47,1.96,1598.56,4010.43,0.52
